# DSPy Optimization — BootstrapFewShot vs MIPROv2

**Week 6 | Notebook 2 of 12**

**What you'll learn:**
- What is compilation? (conceptual walkthrough)
- Preparing trainset + devset (dspy.Example format)
- Writing a custom metric function
- Running BootstrapFewShot — inspect generated few-shot demos
- Running MIPROv2 — inspect generated instructions
- Comparing: baseline vs optimized on devset score
- Saving and reloading the optimized program

**Runtime:** ~45 minutes (API calls during optimization)

**Cost-saving:** Default 10 trials (reduce in .env with DSPY_OPTIMIZER_TRIALS)

In [21]:
# 💰 COST ESTIMATE
import warnings

from src.cost_tracker import print_cost_warning

warnings.filterwarnings("ignore")

print_cost_warning("06_dspy/02_optimizers.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/02_optimizers.ipynb
Task:      DSPy optimizers (expensive)
Calls:     ~60

With GPT-4o:       $0.90 USD
With GPT-4o-mini:  $0.09 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup

In [22]:
import dspy
from dspy.evaluate import Evaluate
from dspy.teleprompt import BootstrapFewShot, MIPROv2

from src.config import DSPY_OPTIMIZER_TRIALS, get_dspy_lm
from src.datasets import generate_qa_pairs

lm = get_dspy_lm()
dspy.configure(lm=lm)

print("✅ DSPy configured")
print(f"Optimizer trials: {DSPY_OPTIMIZER_TRIALS}")

✅ DSPy configured
Optimizer trials: 10


## 2. What Is Compilation?

In [23]:
# DSPy compilation = automatically improving your program
# by generating better instructions and few-shot examples

print("DSPy Compilation Flow:")
print("  1. Define your program (signatures + modules)")
print("  2. Provide training examples")
print("  3. Define a metric (0-1 scoring function)")
print("  4. Run optimizer (BootstrapFewShot / MIPROv2)")
print("  5. Optimized program has better prompts + demos")
print("  6. Evaluate on devset")
print("\n💡 Think of it like a compiler: Python → optimized prompts")

DSPy Compilation Flow:
  1. Define your program (signatures + modules)
  2. Provide training examples
  3. Define a metric (0-1 scoring function)
  4. Run optimizer (BootstrapFewShot / MIPROv2)
  5. Optimized program has better prompts + demos
  6. Evaluate on devset

💡 Think of it like a compiler: Python → optimized prompts


## 3. Preparing Trainset + Devset

In [24]:
# Convert synthetic data to dspy.Example format
qa_data = generate_qa_pairs(40)

examples = [
    dspy.Example(question=d["question"], answer=d["expected"]).with_inputs("question")
    for d in qa_data
]

# Split: 70% train, 30% dev
split = int(len(examples) * 0.7)
trainset = examples[:split]
devset = examples[split:]

print(f"Trainset: {len(trainset)} examples")
print(f"Devset: {len(devset)} examples")
print("\nSample train example:")
print(f"  Question: {trainset[0].question}")
print(f"  Answer: {trainset[0].answer}")

Trainset: 28 examples
Devset: 12 examples

Sample train example:
  Question: What is DSPy?
  Answer: DSPy is a framework for programming language models.


In [25]:
examples

[Example({'question': 'What is DSPy?', 'answer': 'DSPy is a framework for programming language models.'}) (input_keys={'question'}),
 Example({'question': 'What is Vector DB?', 'answer': 'Vector databases store embeddings for similarity search.'}) (input_keys={'question'}),
 Example({'question': 'What is Fine-tuning?', 'answer': 'Fine-tuning adapts a pre-trained model to a specific task.'}) (input_keys={'question'}),
 Example({'question': 'What is DSPy?', 'answer': 'DSPy is a framework for programming language models.'}) (input_keys={'question'}),
 Example({'question': 'What is DSPy?', 'answer': 'DSPy is a framework for programming language models.'}) (input_keys={'question'}),
 Example({'question': 'What is Attention?', 'answer': 'Attention mechanisms let models focus on relevant tokens.'}) (input_keys={'question'}),
 Example({'question': 'What is RAG?', 'answer': 'Retrieval-Augmented Generation combines search with LLMs.'}) (input_keys={'question'}),
 Example({'question': 'What is Fi

## 4. Writing a Custom Metric

In [26]:
def answer_metric(example, prediction, trace=None):
    """Check if predicted answer covers the expected answer's content.

    Word-overlap ratio (expected words found in prediction). A plain substring
    check scores 0 on paraphrases, which leaves optimizers with nothing to
    bootstrap from.
    """
    expected = set(example.answer.lower().split())
    predicted = set(prediction.answer.lower().split())
    if not expected:
        return 0.0
    return 1.0 if len(expected & predicted) / len(expected) >= 0.3 else 0.0


# Test the metric
class TestEx(dspy.Example):
    pass


ex = dspy.Example(question="What is RAG?", answer="Retrieval-Augmented Generation").with_inputs(
    "question"
)
pred = dspy.Prediction(answer="RAG")
print(f"Metric score: {answer_metric(ex, pred)}")

Metric score: 0.0


## 5. Baseline Program

In [27]:
class QA(dspy.Signature):
    """Answer questions with short factual responses."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


class SimpleQA(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(QA)

    def forward(self, question):
        return self.generate(question=question)


baseline = SimpleQA()

# Evaluate baseline
evaluator = Evaluate(devset=devset, metric=answer_metric, num_threads=4, display_progress=True)
baseline_score = evaluator(baseline).score
print(f"\nBaseline score: {baseline_score:.2f}")

Average Metric: 7.00 / 12 (58.3%): 100%|██████████| 12/12 [00:00<00:00, 121.82it/s]

2026/09/20 21:00:36 INFO dspy.evaluate.evaluate: Average Metric: 7.0 / 12 (58.3%)




Baseline score: 58.33


## 6. BootstrapFewShot — Quick Baseline Optimizer

In [28]:
# BootstrapFewShot: generates few-shot demos by running the program
teleprompter = BootstrapFewShot(metric=answer_metric, max_bootstrapped_demos=4)

optimized_bootstrap = teleprompter.compile(SimpleQA(), trainset=trainset)

# Evaluate optimized
bootstrap_score = evaluator(optimized_bootstrap).score
print(f"\nBootstrapFewShot score: {bootstrap_score:.2f}")
print(f"Improvement: {bootstrap_score - baseline_score:+.2f}")

 18%|█▊        | 5/28 [00:00<00:01, 21.57it/s]


Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Average Metric: 10.00 / 12 (83.3%): 100%|██████████| 12/12 [00:00<00:00, 216.33it/s]

2026/09/20 21:34:00 INFO dspy.evaluate.evaluate: Average Metric: 10.0 / 12 (83.3%)




BootstrapFewShot score: 83.33
Improvement: +25.00


In [31]:
#optimized_bootstrap.save("optimized_bootstrap1.json")
optimized_bootstrap(question="What is RAG?")
dspy.inspect_history(n=1)





[2026-09-20T21:55:28.235428]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Answer questions with short factual responses.


User message:

This is an example of the task, though some input or output fields are not supplied.

[[ ## question ## ]]
What is Fine-tuning?


Assistant message:

[[ ## reasoning ## ]]
Not supplied for this particular example. 

[[ ## answer ## ]]
Fine-tuning adapts a pre-trained model to a specific task.

[[ ## completed ## ]]


User message:

This is an example of the task, though some input or output fields are not supplied.

[[ ## question ## ]]
What is DSPy?


Assistant message:

[[ ## reasoning ## ]]


## 7. MIPROv2 — Bayesian Optimization

In [32]:
# MIPROv2: Bayesian optimization of instructions + demos
# More expensive but generally better results

mipro = MIPROv2(
    metric=answer_metric,
    auto=None,  # dspy 3.3: opt out of auto budget to set candidates/trials manually
    num_candidates=5,  # Reduced for cost
    init_temperature=1.0,
)

optimized_mipro = mipro.compile(
    SimpleQA(),
    trainset=trainset,
    num_trials=DSPY_OPTIMIZER_TRIALS,  # Configurable via .env
    valset=devset,
    minibatch=False,  # Small valset (12) — default minibatch size is 35
)

# Evaluate optimized
mipro_score = evaluator(optimized_mipro).score
print(f"\nMIPROv2 score: {mipro_score:.2f}")
print(f"Improvement over baseline: {mipro_score - baseline_score:+.2f}")
print(f"Improvement over Bootstrap: {mipro_score - bootstrap_score:+.2f}")

2026/09/20 22:37:50 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/09/20 22:37:50 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/09/20 22:37:50 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=5 sets of demonstrations...


Bootstrapping set 1/5
Bootstrapping set 2/5
Bootstrapping set 3/5


 18%|█▊        | 5/28 [00:00<00:01, 21.11it/s]


Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Bootstrapping set 4/5


 14%|█▍        | 4/28 [00:00<00:00, 31.08it/s]


...


Average Metric: 9.00 / 12 (75.0%): 100%|██████████| 12/12 [00:00<00:00, 932.05it/s]

2026/09/20 22:37:52 INFO dspy.evaluate.evaluate: Average Metric: 9.0 / 12 (75.0%)
2026/09/20 22:37:52 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 75.0 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 2'].
2026/09/20 22:37:52 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [58.33, 83.33, 83.33, 83.33, 83.33, 75.0, 83.33, 75.0, 58.33, 83.33, 75.0]
2026/09/20 22:37:52 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 83.33
2026/09/20 22:37:52 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/09/20 22:37:52 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 83.33!



Average Metric: 10.00 / 12 (83.3%): 100%|██████████| 12/12 [00:00<00:00, 308.06it/s]

2026/09/20 22:37:52 INFO dspy.evaluate.evaluate: Average Metric: 10.0 / 12 (83.3%)




MIPROv2 score: 83.33
Improvement over baseline: +25.00
Improvement over Bootstrap: +0.00


## 8. Inspecting What the Optimizers Changed

Optimizers return a *compiled program* — but what actually changed inside it? For both
optimizers, everything attaches to the inner `Predict` of the `ChainOfThought`
(`optimized_*.generate.predict`):

- **`.demos`** — the few-shot examples copied into every prompt
- **`.signature.instructions`** — the task text the model reads

The two optimizers differ in *which* of these they touch: BootstrapFewShot only adds
demos; MIPROv2 rewrites **both** instructions and demos.

In [29]:
# What BootstrapFewShot changed: demos attached to the inner Predict
# (optimized_bootstrap.generate is the ChainOfThought; demos live on its .predict)
demos = optimized_bootstrap.generate.predict.demos
print(f"BootstrapFewShot attached {len(demos)} demos")
print("(a mix of self-generated traces that passed the metric (<=4) and")
print("labeled trainset examples (<=16) — the two max_*_demos knobs)\n")
for i, demo in enumerate(demos, 1):
    print(f"--- demo {i} ---")
    print(f"Q: {demo.question}")
    print(f"A: {demo.answer}\n")

print("=" * 50)
print("Instructions — BootstrapFewShot does NOT change these:")
print(optimized_bootstrap.generate.predict.signature.instructions)

BootstrapFewShot attached 16 demos
(a mix of self-generated traces that passed the metric (<=4) and
labeled trainset examples (<=16) — the two max_*_demos knobs)

--- demo 1 ---
Q: What is DSPy?
A: DSPy is tailored PyTorch for specific domain applications.

--- demo 2 ---
Q: What is Fine-tuning?
A: Fine-tuning is adapting a pre-trained model to a specific task using additional data.

--- demo 3 ---
Q: What is DSPy?
A: DSPy is tailored PyTorch for specific domain applications.

--- demo 4 ---
Q: What is DSPy?
A: DSPy is tailored PyTorch for specific domain applications.

--- demo 5 ---
Q: What is Fine-tuning?
A: Fine-tuning adapts a pre-trained model to a specific task.

--- demo 6 ---
Q: What is DSPy?
A: DSPy is a framework for programming language models.

--- demo 7 ---
Q: What is Vector DB?
A: Vector databases store embeddings for similarity search.

--- demo 8 ---
Q: What is Attention?
A: Attention mechanisms let models focus on relevant tokens.

--- demo 9 ---
Q: What is Prompt En

In [11]:
# MIPROv2 rewrites INSTRUCTIONS as well as demos — inspect both artifacts
print("MIPROv2-optimized instructions (rewritten by the optimizer):")
print(optimized_mipro.generate.predict.signature.instructions)

mipro_demos = optimized_mipro.generate.predict.demos
print(f"\nMIPROv2 attached {len(mipro_demos)} demos:")
for i, demo in enumerate(mipro_demos[:2], 1):
    print(f"  demo {i}: {demo.question} -> {demo.answer[:60]}")

MIPROv2-optimized instructions (rewritten by the optimizer):
Imagine you are an AI specialist assisting a team during a critical machine learning deployment meeting. A colleague urgently asks you a technical question that needs a quick yet informative response to ensure the team's success. Use your expertise to provide a concise and factual answer, including the reasoning behind it, to facilitate a clear understanding and aid timely decision-making.

MIPROv2 attached 4 demos:
  demo 1: What is DSPy? -> DSPy is a framework for programming language models.
  demo 2: What is Vector DB? -> Vector databases store embeddings for similarity search.


In [12]:
# See the ACTUAL prompt the LM receives — demos rendered inline.
# Works for any compiled program; here with the BootstrapFewShot one:
optimized_bootstrap(question="What is Attention?")
dspy.inspect_history(n=1)  # the full message with [[ ## question ## ]] / [[ ## answer ## ]] blocks





[2026-09-20T19:28:06.336220]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Answer questions with short factual responses.


User message:

This is an example of the task, though some input or output fields are not supplied.

[[ ## question ## ]]
What is DSPy?


Assistant message:

[[ ## reasoning ## ]]
Not supplied for this particular example. 

[[ ## answer ## ]]
DSPy is a framework for programming language models.

[[ ## completed ## ]]


User message:

This is an example of the task, though some input or output fields are not supplied.

[[ ## question ## ]]
What is DSPy?


Assistant message:

[[ ## reasoning ## ]]
Not supplied 

## 9. Saving and Reloading

In [33]:
# Save optimized program
optimized_mipro.save("optimized_qa.json")
print("✅ Saved to optimized_qa.json")

# Reload
loaded = SimpleQA()
loaded.load("optimized_qa.json")

# Verify it works
result = loaded(question="What is DSPy?")
print(f"\nReloaded program answer: {result.answer}")

✅ Saved to optimized_qa.json

Reloaded program answer: DSPy is a framework for programming language models within distributed systems.


## 10. Exercise: Optimize a Classification Pipeline

Optimize a sentiment classifier on your own dataset using:
1. BootstrapFewShot as baseline
2. MIPROv2 for best results
3. Compare scores and inspect generated demos

In [14]:
# YOUR TURN: Optimize a classification pipeline

# class Sentiment(dspy.Signature):
#     """Classify sentiment."""
#     text: str = dspy.InputField()
#     sentiment: str = dspy.OutputField()

# class SentimentClassifier(dspy.Module):
#     def __init__(self):
#         super().__init__()
#         self.classify = dspy.ChainOfThought(Sentiment)
#     def forward(self, text):
#         return self.classify(text=text)

# # Optimize
# teleprompter = MIPROv2(metric=your_metric)
# optimized = teleprompter.compile(SentimentClassifier(), trainset=trainset, num_trials=10)

---

**Next:** [03_rag_pipeline.ipynb](03_rag_pipeline.ipynb) — RAG with assertions and constraints